# Instrument a Python LLM Pipeline with OpenTelemetry

Companion notebook for the [Production Observability for LLMs](https://codemia.io/courses/applied_ai_systems/production_observability_for_llms) lesson on Codemia.

**What you will build**: a small RAG pipeline (retrieve, call a model, call a tool, call the
model again) wrapped in OpenTelemetry spans and exported to a local Jaeger instance. You will read
the trace tree for one user query, both as text in the notebook and from Jaeger's own API, and
then break the trace on purpose in the two ways production traces usually break.

## Setup

Jaeger runs as a single binary with in-memory storage: it accepts OTLP on port 4318 and serves
its UI and API on port 16686. The next cells download the binary for your platform and start it
in the background. Docker would also work, but Colab has no Docker.

With an `OPENAI_API_KEY` set the model calls are real `gpt-4o-mini` calls with real token counts.
Without one, a scripted stand-in returns the same shapes; the spans, attributes and tree come out
the same either way.

In [ ]:
%pip install -q opentelemetry-sdk opentelemetry-exporter-otlp-proto-http openai numpy

In [ ]:
import json
import os
import platform
import subprocess
import tarfile
import time
import urllib.request

import numpy as np

try:
    from google.colab import userdata
    os.environ.setdefault('OPENAI_API_KEY', userdata.get('OPENAI_API_KEY') or '')
except Exception:
    pass

USE_REAL_LLM = bool(os.environ.get('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
JAEGER_UI = 'http://localhost:16686'
print('llm:', MODEL if USE_REAL_LLM else 'scripted stand-in')

In [ ]:
JAEGER_VERSION = '2.22.0'


def jaeger_up():
    try:
        urllib.request.urlopen(JAEGER_UI + '/api/v3/services', timeout=2)
        return True
    except Exception:
        return False


def start_jaeger():
    if jaeger_up():
        return 'already running'
    osname = {'Linux': 'linux', 'Darwin': 'darwin'}[platform.system()]
    arch = {'x86_64': 'amd64', 'AMD64': 'amd64', 'arm64': 'arm64', 'aarch64': 'arm64'}[platform.machine()]
    name = f'jaeger-{JAEGER_VERSION}-{osname}-{arch}'
    if not os.path.exists(f'/tmp/{name}/jaeger'):
        url = f'https://github.com/jaegertracing/jaeger/releases/download/v{JAEGER_VERSION}/{name}.tar.gz'
        urllib.request.urlretrieve(url, f'/tmp/{name}.tar.gz')
        tarfile.open(f'/tmp/{name}.tar.gz').extractall('/tmp')
    # No config file means the default all-in-one setup: OTLP in, memory storage, UI on 16686.
    subprocess.Popen([f'/tmp/{name}/jaeger'], stdout=open('/tmp/jaeger.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(30):
        if jaeger_up():
            return 'started'
        time.sleep(1)
    raise RuntimeError('Jaeger did not come up; see /tmp/jaeger.log')


print('jaeger:', start_jaeger())

## Concept recap

A **trace** is one request's journey; a **span** is one timed operation inside it, with a name,
a start and end, attributes, and a parent. The tree of parents is what makes a trace readable:
a slow request becomes "the second model call took 2.1 s of the 2.6 s", which is something you
can act on.

For LLM work, the OpenTelemetry GenAI semantic conventions name the attributes that matter:
`gen_ai.request.model`, `gen_ai.usage.input_tokens`, `gen_ai.usage.output_tokens`,
`gen_ai.tool.name`, and so on. Using the standard names means any backend that understands them
(Jaeger, Grafana, Honeycomb, Datadog) can chart them without custom parsing.

## Tracer setup

Two exporters on one provider: OTLP over HTTP to Jaeger, and an in-memory exporter so the notebook
can print the tree itself. The `service.name` resource attribute is what Jaeger lists in its
service dropdown.

In [ ]:
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode

SERVICE = 'rag-notebook'
memory = InMemorySpanExporter()
provider = TracerProvider(resource=Resource.create({'service.name': SERVICE}))
provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter('http://localhost:4318/v1/traces')))
provider.add_span_processor(SimpleSpanProcessor(memory))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer('rag.pipeline')

## The pipeline

A support assistant for a made-up shop. Retrieval is bag-of-words cosine similarity over six
help-centre snippets (real embeddings would be one more span). The model has one tool,
`get_order_status`, so a question about an order produces the full pattern: model call, tool
call, second model call.

In [ ]:
DOCS = [
    'Returns are accepted within 30 days of delivery for unused items in original packaging.',
    'Refunds are issued to the original payment method within 5 business days of receiving a return.',
    'Standard shipping takes 3 to 5 business days. Express shipping takes 1 to 2 business days.',
    'Orders can be cancelled free of charge until they leave the warehouse.',
    'Gift cards cannot be returned or exchanged for cash.',
    'Support is available by chat from 8am to 8pm UTC on weekdays.',
]
VOCAB = sorted({w for d in DOCS for w in d.lower().replace('.', '').split()})


def embed(text):
    words = text.lower().replace('?', '').replace('.', '').split()
    v = np.array([words.count(w) for w in VOCAB], dtype=float)
    return v / (np.linalg.norm(v) or 1)


DOC_VECS = np.stack([embed(d) for d in DOCS])
ORDERS = {'1042': {'status': 'delivered', 'delivered_on': '2026-10-02'},
          '1043': {'status': 'in transit', 'eta': '2026-10-18'}}

In [ ]:
def retrieve(query, k=2):
    with tracer.start_as_current_span('retrieve') as span:
        with tracer.start_as_current_span('embed_query') as s:
            q = embed(query)
            s.set_attribute('embedding.dimensions', len(q))
        with tracer.start_as_current_span('vector_search') as s:
            scores = DOC_VECS @ q
            top = np.argsort(-scores)[:k]
            s.set_attribute('search.k', k)
            s.set_attribute('search.top_scores', [round(float(scores[i]), 3) for i in top])
        span.set_attribute('retrieval.document_ids', [int(i) for i in top])
        return [DOCS[i] for i in top]


def get_order_status(order_id):
    with tracer.start_as_current_span('execute_tool get_order_status') as span:
        span.set_attribute('gen_ai.operation.name', 'execute_tool')
        span.set_attribute('gen_ai.tool.name', 'get_order_status')
        span.set_attribute('order.id', order_id)
        time.sleep(0.05)  # stands in for a database round trip
        if order_id not in ORDERS:
            raise KeyError(f'no order {order_id}')
        return ORDERS[order_id]

In [ ]:
TOOLS = [{'type': 'function', 'function': {
    'name': 'get_order_status', 'description': 'Look up the delivery status of an order by id.',
    'parameters': {'type': 'object', 'properties': {'order_id': {'type': 'string'}},
                   'required': ['order_id']}}}]


def scripted_model(messages):
    # Ask for the tool once if an order number appears, then answer from what is in context.
    import re
    user = messages[1]['content']
    tool_results = [m['content'] for m in messages if m['role'] == 'tool']
    order = re.search(r'\b(\d{4})\b', user)
    if order and not tool_results:
        return None, [('call_0', 'get_order_status', {'order_id': order.group(1)})], (180, 20)
    facts = ' '.join(tool_results) + ' ' + messages[0]['content'].split('Context:')[-1]
    return f'Based on our records: {facts.strip()[:200]}', [], (240, 45)

In [ ]:
def chat(messages):
    with tracer.start_as_current_span(f'chat {MODEL}') as span:
        span.set_attribute('gen_ai.operation.name', 'chat')
        span.set_attribute('gen_ai.system', 'openai')
        span.set_attribute('gen_ai.request.model', MODEL)
        if USE_REAL_LLM:
            from openai import OpenAI
            r = OpenAI().chat.completions.create(model=MODEL, messages=messages, tools=TOOLS,
                                                 temperature=0)
            m = r.choices[0].message
            calls = [(c.id, c.function.name, json.loads(c.function.arguments)) for c in m.tool_calls or []]
            text, usage = m.content, (r.usage.prompt_tokens, r.usage.completion_tokens)
            span.set_attribute('gen_ai.response.finish_reasons', [r.choices[0].finish_reason])
        else:
            text, calls, usage = scripted_model(messages)
            span.set_attribute('demo.token_counts_are_made_up', True)
        span.set_attribute('gen_ai.usage.input_tokens', usage[0])
        span.set_attribute('gen_ai.usage.output_tokens', usage[1])
        return text, calls

In [ ]:
def answer(question):
    with tracer.start_as_current_span('rag.query') as root:
        root.set_attribute('user.query', question)
        context = retrieve(question)
        messages = [{'role': 'system', 'content': 'You are a shop support assistant. Answer briefly '
                     'using the context. When the user mentions an order number, look it up with '
                     'get_order_status before answering.\nContext: ' + ' '.join(context)},
                    {'role': 'user', 'content': question}]
        text, calls = chat(messages)
        if calls:
            messages.append({'role': 'assistant', 'content': None, 'tool_calls': [
                {'id': cid, 'type': 'function', 'function': {'name': n, 'arguments': json.dumps(a)}}
                for cid, n, a in calls]})
            for cid, name, args in calls:
                result = get_order_status(**args)
                messages.append({'role': 'tool', 'tool_call_id': cid, 'content': json.dumps(result)})
            text, _ = chat(messages)
        root.set_attribute('response.length', len(text or ''))
        return text, root.get_span_context().trace_id

## Experiment: one query, one trace

Run a question that needs both retrieval and the tool, then print the tree from the in-memory
exporter: indentation is parenthood, the bar is where each span sits on the request's timeline.

In [ ]:
memory.clear()
text, trace_id = answer('I want to send back order 1042. When did it arrive, and how long do I have to return it?')
print('answer:', text)
print('trace id:', format(trace_id, '032x'))

In [ ]:
def print_tree(spans, width=40):
    by_parent = {}
    for s in spans:
        by_parent.setdefault(s.parent.span_id if s.parent else None, []).append(s)
    t0 = min(s.start_time for s in spans)
    total = max(s.end_time for s in spans) - t0

    def walk(parent, depth):
        for s in sorted(by_parent.get(parent, []), key=lambda s: s.start_time):
            a = int((s.start_time - t0) / total * width)
            b = max(a + 1, int((s.end_time - t0) / total * width))
            bar = ' ' * a + '#' * (b - a) + ' ' * (width - b)
            flag = '  ERROR' if s.status.status_code == StatusCode.ERROR else ''
            print(f'{"  " * depth + s.name:<36} {(s.end_time - s.start_time) / 1e6:8.1f} ms |{bar}|{flag}')
            walk(s.context.span_id, depth + 1)
    walk(None, 0)


print_tree(memory.get_finished_spans())

In [ ]:
# The attributes on each model call: what a cost dashboard would sum.
for s in memory.get_finished_spans():
    if s.name.startswith('chat'):
        a = s.attributes
        print(s.name, '| in', a['gen_ai.usage.input_tokens'], '| out', a['gen_ai.usage.output_tokens'])

### The same trace, from Jaeger

Flush the batch exporter, then ask Jaeger's HTTP API for the trace by id. Jaeger 2 answers in
OTLP JSON, the same format the SDK exported, so the parent links are plain `parentSpanId` fields.
If this prints the same tree, the export worked and the UI at `localhost:16686` will show it too.

In [ ]:
def jaeger_tree(trace_id):
    provider.force_flush()
    tid = format(trace_id, '032x')
    spans = []
    for _ in range(10):  # the batch exporter and Jaeger's ingest are both asynchronous
        try:
            data = json.load(urllib.request.urlopen(f'{JAEGER_UI}/api/v3/traces/{tid}'))
            spans = [s for rs in data['result']['resourceSpans']
                     for ss in rs['scopeSpans'] for s in ss['spans']]
            break
        except Exception:
            time.sleep(1)

    def walk(pid, depth):
        kids = [s for s in spans if s.get('parentSpanId') == pid]
        for s in sorted(kids, key=lambda s: int(s['startTimeUnixNano'])):
            ms = (int(s['endTimeUnixNano']) - int(s['startTimeUnixNano'])) / 1e6
            print(f'{"  " * depth + s["name"]:<36} {ms:8.1f} ms')
            walk(s['spanId'], depth + 1)
    walk(None, 0)
    return len(spans)


print(jaeger_tree(trace_id), 'spans stored in Jaeger')

In [ ]:
# In Colab, open the Jaeger UI in a new tab. Locally, just open http://localhost:16686
try:
    from google.colab import output
    output.serve_kernel_port_as_window(16686)
except Exception:
    print(f'Open {JAEGER_UI}/trace/{format(trace_id, "032x")}')

## Variations

Two ways traces break in production, and one way to make them cheaper.

In [ ]:
# 1. A failing tool. Record the exception on the span and mark it as an error, so the failure
# is visible in the tree instead of only in a log file somewhere else.
memory.clear()
with tracer.start_as_current_span('rag.query') as root:
    try:
        get_order_status('9999')
    except KeyError as e:
        root.record_exception(e)
        root.set_status(Status(StatusCode.ERROR, str(e)))
print_tree(memory.get_finished_spans())
events = [ev.name for s in memory.get_finished_spans() for ev in s.events]
print('events recorded:', events)

Both spans are marked, and only the root's was done by hand. A span started with
`start_as_current_span` records an exception that escapes its `with` block and sets its own status
to ERROR, so the tool span flagged itself before the exception reached our `except`. Hand-written
handling matters at the boundaries: here the root caught the error, so without the two explicit
lines the root would look like a success even though its request failed.

In [ ]:
for s in memory.get_finished_spans():
    print(f'{s.name:<32} status={s.status.status_code.name:<6} events={[e.name for e in s.events]}')

In [ ]:
# 2. Lost context across threads. Run the tool in a thread pool. The worker thread does not
# inherit the current span, so the tool span starts a brand new trace: an orphan.
from concurrent.futures import ThreadPoolExecutor
from opentelemetry import context as otel_context

memory.clear()
with tracer.start_as_current_span('rag.query'):
    with ThreadPoolExecutor(1) as pool:
        pool.submit(get_order_status, '1043').result()
        ctx = otel_context.get_current()  # capture it, then attach it inside the worker
        pool.submit(lambda: (otel_context.attach(ctx), get_order_status('1043'))).result()

spans = memory.get_finished_spans()
for s in spans:
    print(f'{s.name:<32} trace={format(s.context.trace_id, "032x")[:8]} parent={"yes" if s.parent else "none"}')
print('distinct traces:', len({s.context.trace_id for s in spans}))

In [ ]:
# 3. Sampling. Keep 30% of traces, decided once at the root and inherited by every child, so a
# trace is either complete or absent, never half there.
from opentelemetry.sdk.trace.sampling import ParentBased, TraceIdRatioBased

sampled = InMemorySpanExporter()
p2 = TracerProvider(sampler=ParentBased(TraceIdRatioBased(0.3)),
                    resource=Resource.create({'service.name': SERVICE + '-sampled'}))
p2.add_span_processor(SimpleSpanProcessor(sampled))
t2 = p2.get_tracer('rag.sampled')
for i in range(50):
    with t2.start_as_current_span('rag.query'):
        with t2.start_as_current_span('retrieve'):
            pass
traces = {s.context.trace_id for s in sampled.get_finished_spans()}
print(f'kept {len(traces)} of 50 traces, {len(sampled.get_finished_spans())} spans '
      f'(always 2 per kept trace)')

Things worth noticing:

1. **The tree answers "where did the time go" at a glance.** In live mode the two `chat` spans
   dominate (about 0.8 s and 0.6 s in our run) while retrieval and the tool take milliseconds.
   That is the usual shape of an LLM request, and why latency work starts with the model calls.
   With the stand-in the model calls are instant, so the tool's simulated database round trip
   is the widest bar instead.
2. **Token counts belong on the span.** With `gen_ai.usage.*` on every model call, cost per
   request is a sum over the trace, and cost per feature is a group-by on any attribute you add
   at the root, like a tenant or an endpoint.
3. **Broken traces fail silently.** The orphaned tool span in variation 2 still reached Jaeger;
   it just sat in its own trace where nobody would look. Context propagation across threads,
   queues and services is where most real instrumentation work goes.

## What to try next

- Add a span attribute for the prompt template version, then change the system prompt and
  compare token counts by version in Jaeger's search.
- Record the prompt and completion as span events (`gen_ai.content.prompt`), then think about
  what that means for user data in your trace store. The GenAI conventions keep content opt-in
  for exactly that reason.
- Put the tool behind a real HTTP call (a tiny Flask app in a thread) and use the
  `opentelemetry-instrumentation-requests` package so the context crosses the network.